## 도전과제 2. session_id 전략 비교 — 어느 게 맞을까?

> **이론 매핑**: Q5. "같은 사용자가 다른 페르소나로 대화하면?"

같은 사람(`user_42`)이 **분석가**와 **마케터** 두 페르소나로 따로따로 대화한다고 해봅시다.  
세 가지 session_id 설계 전략 중 *어느 게 옳은가* 판단해보세요.

- **A)** `session_id = user_id` — 페르소나는 무시
- **B)** `session_id = f"{user_id}_{persona}"` — 사용자 × 페르소나 조합 ⭐ 정답일까?
- **C)** `session_id = persona` — 사용자는 무시

세 전략을 *같은 시나리오*에 흘려보내서, 어느 경우에 *대화가 섞이는지* 직접 확인하세요.

In [2]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 로드 완료" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, openai_api_key=os.environ.get("OPENAI_API_KEY"))
print("✅ LLM 준비 완료")

✅ 로컬 Jupyter 환경입니다.
✅ API 키 로드 완료
✅ LLM 준비 완료


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [5]:
# 📌 페르소나 + 메모리 챗봇 — 빈칸을 채우세요
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 👇 빈칸 1: 프롬프트 — system에 {persona}, history 자리, human에 {question}
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나의 말투와 태도를 일관되게 유지하세요."),
    MessagesPlaceholder(variable_name="history"),   # ← 대화 기록이 들어갈 자리
    ("human", "{question}"),
])

# 세션 저장소 (재사용)
persona_store = {}
def get_persona_history(session_id):
    if session_id not in persona_store:
        persona_store[session_id] = InMemoryChatMessageHistory()
    return persona_store[session_id]

# 👇 빈칸 2: 체인을 메모리로 감싸기
persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",   # ← 매번 새로 들어오는 사용자 입력 키
    history_messages_key="history",  # ← 위 MessagesPlaceholder와 같은 이름!
)


def chat_with_persona(session_id: str, persona: str, question: str) -> str:
    """페르소나·세션·질문을 받아 답변 반환"""
    result = persona_chain.invoke(
        {"persona": persona, "question": question},
        config={"configurable": {"session_id": session_id}},
    )
    return result.content


# 테스트 — 같은 세션에서 3턴 대화
sid = "test_user"
persona = "냉정하지만 친절한 시니어 데이터 분석가"
print("[1턴]", chat_with_persona(sid, persona, "안녕하세요. 저는 진환이라고 합니다."))
print("[2턴]", chat_with_persona(sid, persona, "표본 분산과 모분산의 차이를 한 줄로 알려주세요."))
print("[3턴]", chat_with_persona(sid, persona, "제가 누구라고 했었죠?"))

c:\Users\user\anaconda3\envs\day01_venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


[1턴] 안녕하세요, 진환님. 만나서 반갑습니다. 데이터 분석에 대해 궁금한 점이나 도움이 필요하신 부분이 있으신가요?
[2턴] 표본 분산은 표본 데이터의 변동성을 측정하는 반면, 모분산은 전체 모집단의 변동성을 나타냅니다.
[3턴] 진환님이라고 말씀하셨습니다. 추가로 궁금한 점이 있으신가요?


In [6]:
# 🎯 3가지 session_id 전략 — 같은 시나리오에 흘려서 결과 비교

def run_strategy(make_sid):
    """주어진 session_id 전략으로 시나리오 실행 후 마지막 답변 반환"""
    # 매 전략마다 새 store + 새 chain (전략 간 격리)
    local_store = {}
    def local_history(sid):
        if sid not in local_store:
            local_store[sid] = InMemoryChatMessageHistory()
        return local_store[sid]

    local_chain = RunnableWithMessageHistory(
        persona_prompt | llm,
        local_history,
        input_messages_key="question",
        history_messages_key="history",
    )

    USER = "user_42"

    # 1️⃣ 분석가 페르소나로 KPI 얘기
    local_chain.invoke(
        {"persona": "꼼꼼한 데이터 분석가", "question": "저는 KPI 대시보드 만드는 일을 합니다."},
        config={"configurable": {"session_id": make_sid(USER, "analyst")}},
    )

    # 2️⃣ 마케터 페르소나로 캠페인 얘기 (다른 페르소나)
    local_chain.invoke(
        {"persona": "톡톡 튀는 마케터", "question": "저는 신제품 캠페인을 준비하고 있어요."},
        config={"configurable": {"session_id": make_sid(USER, "marketer")}},
    )

    # 3️⃣ 다시 분석가 페르소나로 "내가 뭘 한다고 했지?" — 분석가 얘기만 기억해야 정상
    answer = local_chain.invoke(
        {"persona": "꼼꼼한 데이터 분석가", "question": "제가 뭘 한다고 했었죠?"},
        config={"configurable": {"session_id": make_sid(USER, "analyst")}},
    )
    return answer.content


strategies = {
    "A (user_id만 사용)":     lambda uid, persona: uid,
    "B (user_id + persona)":  lambda uid, persona: f"{uid}_{persona}",
    "C (persona만 사용)":     lambda uid, persona: persona,
}

for name, make_sid in strategies.items():
    print(f"\n{'='*60}")
    print(f"🧪 전략: {name}")
    print('='*60)
    print(run_strategy(make_sid))


🧪 전략: A (user_id만 사용)


C:\Users\user\AppData\Local\Temp\ipykernel_28736\3342924622.py:51: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  print(run_strategy(make_sid))


당신은 KPI 대시보드를 만드는 일을 하고 있으며, 현재 신제품 캠페인을 준비하고 있다고 말씀하셨습니다. 두 가지 모두 중요한 역할을 하며, 데이터 분석과 성과 측정이 캠페인의 성공에 큰 영향을 미칠 것입니다. 추가적으로 궁금한 점이나 더 논의하고 싶은 사항이 있으신가요?

🧪 전략: B (user_id + persona)


C:\Users\user\AppData\Local\Temp\ipykernel_28736\3342924622.py:51: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  print(run_strategy(make_sid))


당신은 KPI 대시보드를 만드는 일을 하고 있다고 말씀하셨습니다. KPI 대시보드는 조직의 성과를 측정하고 모니터링하는 데 중요한 역할을 합니다. 이와 관련하여 더 구체적인 질문이나 논의하고 싶은 사항이 있다면 말씀해 주세요.

🧪 전략: C (persona만 사용)


C:\Users\user\AppData\Local\Temp\ipykernel_28736\3342924622.py:51: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  print(run_strategy(make_sid))


당신은 KPI 대시보드를 만드는 일을 하고 있다고 말씀하셨습니다. KPI 대시보드는 성과 지표를 시각적으로 표현하여 비즈니스의 성과를 모니터링하고 분석하는 데 중요한 역할을 합니다. 추가적으로 궁금한 점이나 도움이 필요한 부분이 있다면 말씀해 주세요.


### 🤔 판단해보기

세 전략의 *분석가 페르소나 답변*을 비교한 뒤 2-3줄로 답해보세요:

1. 분석가가 *KPI 대시보드*만 기억하게 한 전략은 어느 것인가? (A / B / C)
2. *마케터 캠페인 얘기*가 분석가 답변에 섞여 들어간 전략은? 왜 그렇게 됐을까?
3. C 전략(persona만)은 *user_42 말고 user_99*도 동시에 쓰면 무슨 일이 일어날까?

> 💡 **이 연습의 진짜 의미**: session_id 설계는 *대화 격리의 단위*를 정하는 일입니다.  
> 잘못 설계하면 **대화 누설(privacy leak)** 이나 맥락 혼선이 발생해요.  
> 실서비스에서 가장 흔하면서도 사고가 자주 나는 지점이라, 손으로 한 번 *섞이는 걸* 봐두는 게 중요합니다.